In [2]:
!git clone https://github.com/AutoCS-wyh/Automotive-cyber-threat-intelligence-corpus.git


Cloning into 'Automotive-cyber-threat-intelligence-corpus'...
remote: Enumerating objects: 3083, done.
remote: Counting objects: 100% (553/553), done.
remote: Compressing objects: 100% (543/543), done.
remote: Total 3083 (delta 145), reused 61 (delta 8), pack-reused 2530 (from 1)
Receiving objects: 100% (3083/3083), 1.34 MiB | 12.06 MiB/s, done.
Resolving deltas: 100% (467/467), done.


In [3]:
!pip install -q transformers seqeval torchcrf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [4]:
!pip install -q "transformers==4.45.2" "tokenizers==0.20.1" "accelerate>=0.21.0" seqeval torchcrf


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 87.0 MB/s eta 0:00:00


In [5]:
!pip install -q "transformers==4.45.2" "tokenizers==0.20.1" "accelerate>=0.21.0" seqeval torchcrf


In [6]:
import os
import random
import numpy as np

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


from seqeval.metrics import f1_score, classification_report

from transformers import AutoTokenizer, AutoModel

In [7]:
!pip install scikit-learn

In [8]:
!pip install -q transformers seqeval torchcrf

In [9]:
!pip install --no-cache-dir git+https://github.com/kmkurn/pytorch-crf.git

  Cloning https://github.com/kmkurn/pytorch-crf.git to /tmp/pip-req-build-qr6zcbja
  Running command git clone --filter=blob:none --quiet https://github.com/kmkurn/pytorch-crf.git /tmp/pip-req-build-qr6zcbja
  Resolved https://github.com/kmkurn/pytorch-crf.git to commit 623e3402d00a2728e99d6e8486010d67c754267b
  Preparing metadata (setup.py) ... done
  Created wheel for pytorch-crf: filename=pytorch_crf-0.7.2-py3-none-any.whl size=6410 sha256=e04d76c7a83ec14bc99a304a72e2c6dedb9372d2dcb2ecfc1ade2a596e07fd8a
  Stored in directory: /tmp/pip-ephem-wheel-cache-gsqe6ckl/wheels/a6/3f/e9/a3eb0981973bf404ca2b280c6b6a885a310174436d0adb3445
Successfully built pytorch-crf


In [10]:
from torchcrf import CRF
print("CRF imported successfully!")

CRF imported successfully!


In [11]:
!find Automotive-cyber-threat-intelligence-corpus -maxdepth 4 -type f

Automotive-cyber-threat-intelligence-corpus/.git/config
Automotive-cyber-threat-intelligence-corpus/.git/objects/pack/pack-00bf4e2885eaad9875529f899fe0305dcd6e3833.idx
Automotive-cyber-threat-intelligence-corpus/.git/objects/pack/pack-00bf4e2885eaad9875529f899fe0305dcd6e3833.pack
Automotive-cyber-threat-intelligence-corpus/.git/index
Automotive-cyber-threat-intelligence-corpus/.git/refs/heads/main
Automotive-cyber-threat-intelligence-corpus/.git/HEAD
Automotive-cyber-threat-intelligence-corpus/.git/hooks/update.sample
Automotive-cyber-threat-intelligence-corpus/.git/hooks/pre-rebase.sample
Automotive-cyber-threat-intelligence-corpus/.git/hooks/commit-msg.sample
Automotive-cyber-threat-intelligence-corpus/.git/hooks/pre-merge-commit.sample
Automotive-cyber-threat-intelligence-corpus/.git/hooks/fsmonitor-watchman.sample
Automotive-cyber-threat-intelligence-corpus/.git/hooks/prepare-commit-msg.sample
Automotive-cyber-threat-intelligence-corpus/.git/hooks/push-to-checkout.sample
Automotive

In [12]:
!head -20 Automotive-cyber-threat-intelligence-corpus/model/BERT-BiLSTM-att-CRF/data1/train.txt

During brute force: powercycling of the ECU to reduce the brute force time because of sending intervall .	O B_AP_hasImpact_1 O O O O S_Com O O O B_AP E_AP O O O O O O
Exploitation of weak secret keys by cryptoanalysis of the Megamos cipher .	O O B_Vul_M_1 I_Vul_M_1 E_Vul_M_1 O S_AP_targets_1 O O B_Com_targets_2 E_Com_targets_2 O
This kind of objects can cause the Tesla Model X (HW 2.5) to brake suddenly or deviate to a different lane .	O O O O O O O B_Veh_consists-of_2 I_Veh_consists-of_2 E_Veh_consists-of_2 O O O B_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 E_Con_hasImpact_2 O
Theft alert is forwarded to thief .	B_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 E_Con_hasImpact_2 O
Bleeding the brakes by spoofing a diagnotic CAN message when vehicle is slower than 5 mph .	B_Con_hasImpact_2 I_Con_hasImpact_2 E_Con_hasImpact_2 O S_AP_M_1 O B_Com_targets_2 I_Com_targets

In [13]:
!head -20 Automotive-cyber-threat-intelligence-corpus/model/BERT-BiLSTM-att-CRF/data1/BIOES.txt

Police O
issue O
warning O
after O
keyless B_Con
car I_Con
theft E_Con
in O
Middlesbrough S_Loc
, O
UK S_Loc
. O

The O
theft O
happened O
on O
Saxonfield S_Loc
, O
Colby B_Loc


In [14]:
!head -50 Automotive-cyber-threat-intelligence-corpus/model/BERT-BiLSTM-att-CRF/data1/train.txt


During brute force: powercycling of the ECU to reduce the brute force time because of sending intervall .	O B_AP_hasImpact_1 O O O O S_Com O O O B_AP E_AP O O O O O O
Exploitation of weak secret keys by cryptoanalysis of the Megamos cipher .	O O B_Vul_M_1 I_Vul_M_1 E_Vul_M_1 O S_AP_targets_1 O O B_Com_targets_2 E_Com_targets_2 O
This kind of objects can cause the Tesla Model X (HW 2.5) to brake suddenly or deviate to a different lane .	O O O O O O O B_Veh_consists-of_2 I_Veh_consists-of_2 E_Veh_consists-of_2 O O O B_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 E_Con_hasImpact_2 O
Theft alert is forwarded to thief .	B_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 I_Con_hasImpact_2 E_Con_hasImpact_2 O
Bleeding the brakes by spoofing a diagnotic CAN message when vehicle is slower than 5 mph .	B_Con_hasImpact_2 I_Con_hasImpact_2 E_Con_hasImpact_2 O S_AP_M_1 O B_Com_targets_2 I_Com_targets

In [15]:
# Cell 2: Paths & reproducibility

BASE_DIR = "Automotive-cyber-threat-intelligence-corpus"

TRAIN_PATH = os.path.join(BASE_DIR,
    "model/BERT-BiLSTM-att-CRF/data1/train.txt")
TEST_PATH  = os.path.join(BASE_DIR,
    "model/BERT-BiLSTM-att-CRF/data1/dev.txt")

print("Train file:", TRAIN_PATH)
print("Test file:", TEST_PATH)

# Reproducibility
SEED = 42
import random, torch, numpy as np
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Train file: Automotive-cyber-threat-intelligence-corpus/model/BERT-BiLSTM-att-CRF/data1/train.txt
Test file: Automotive-cyber-threat-intelligence-corpus/model/BERT-BiLSTM-att-CRF/data1/dev.txt


device(type='cuda')

In [26]:
# Cell 3: Read ACTI joint BIOES data (token-level sequences)

def read_acti_joint(path):
    """
    Reads ACTI train/dev files where each line is:
        token1 token2 ... tokenN \t label1 label2 ... labelN
    Returns: list of (tokens, labels)
    """
    sentences = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # must split into EXACTLY two parts
            if "\t" not in line:
                continue  # skip malformed lines

            tokens_str, labels_str = line.split("\t")
            tokens = tokens_str.split()
            labels = labels_str.split()

            # validate lengths
            if len(tokens) != len(labels):
                print("Warning: length mismatch:")
                print("Tokens:", tokens)
                print("Labels:", labels)
                continue

            sentences.append((tokens, labels))

    return sentences


train_sentences = read_acti_joint(TRAIN_PATH)
test_sentences  = read_acti_joint(TEST_PATH)

print("Train sentences:", len(train_sentences))
print("Test sentences :", len(test_sentences))

# peek first sample
train_sentences[0]


Train sentences: 2426
Test sentences : 606


(['During',
  'brute',
  'force:',
  'powercycling',
  'of',
  'the',
  'ECU',
  'to',
  'reduce',
  'the',
  'brute',
  'force',
  'time',
  'because',
  'of',
  'sending',
  'intervall',
  '.'],
 ['O',
  'B_AP_hasImpact_1',
  'O',
  'O',
  'O',
  'O',
  'S_Com',
  'O',
  'O',
  'O',
  'B_AP',
  'E_AP',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'])

In [51]:
# Cell 4: Build label ↔ id mapping (ENTITY-ONLY, paper-aligned)

def simplify_label(label):
    if label == "O":
        return "O"
    prefix, rest = label.split("_", 1)
    entity = rest.split("_")[0]   # keep only entity type
    if prefix == "S":
        prefix = "B"
    elif prefix == "E":
        prefix = "I"
    return f"{prefix}-{entity}"

labels_set = set()

for _, labels in train_sentences + test_sentences:
    for lab in labels:
        labels_set.add(simplify_label(lab))

labels_list = sorted(labels_set)

label2id = {lab: i for i, lab in enumerate(labels_list)}
id2label = {i: lab for lab, i in label2id.items()}

num_labels = len(labels_list)

print("Number of labels (simplified):", num_labels)
print("Labels:", labels_list)


Number of labels (simplified): 21
Labels: ['B-AP', 'B-AV', 'B-CoA', 'B-Com', 'B-Con', 'B-Ide', 'B-Loc', 'B-Tool', 'B-Veh', 'B-Vul', 'I-AP', 'I-AV', 'I-CoA', 'I-Com', 'I-Con', 'I-Ide', 'I-Loc', 'I-Tool', 'I-Veh', 'I-Vul', 'O']


In [52]:
# Cell 5: Tokenizer + Dataset class (WITH label simplification)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

MAX_LEN = 128

class ActiDataset(torch.utils.data.Dataset):
    def __init__(self, sentences, tokenizer, label2id, max_len=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        tokens, labels = self.sentences[idx]

        encoding = tokenizer(
            tokens,
            is_split_into_words=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        word_ids = encoding.word_ids(batch_index=0)

        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            else:
                label_ids.append(label2id[simplify_label(labels[word_id])])

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(label_ids),
            "word_ids": word_ids,
            "tokens": tokens,
            "word_labels": labels
        }

def collate_fn(batch):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in batch]).to(device),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]).to(device),
        "labels": torch.stack([x["labels"] for x in batch]).to(device),
        "word_ids_list": [x["word_ids"] for x in batch],
        "tokens_list": [x["tokens"] for x in batch],
        "word_labels_list": [x["word_labels"] for x in batch],
    }

train_dataset = ActiDataset(train_sentences, tokenizer, label2id, MAX_LEN)
test_dataset  = ActiDataset(test_sentences, tokenizer, label2id, MAX_LEN)

len(train_dataset), len(test_dataset)


(2426, 606)

In [53]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16
EPOCHS = 40        # required for BiLSTM to learn
LR = 1e-3          # MUST be higher than BERT LR

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

print("Train batches:", len(train_loader))
print("Test batches :", len(test_loader))


Train batches: 152
Test batches : 38


In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [54]:
import torch
from torch import nn

class BiLstmAttLstm(nn.Module):
    """
    BiLSTM-att-LSTM:
    Encoder: BiLSTM over token embeddings
    Attention: scalar self-attention over encoder outputs
    Decoder: LSTM -> classifier (softmax via CrossEntropyLoss)
    """
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        embed_dim: int = 300,
        enc_hidden: int = 300,     # BiLSTM_dim in paper family is often 300 for non-BERT variants
        dec_hidden: int = 300,
        dropout: float = 0.1,
        pad_token_id: int = 0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token_id)

        self.encoder = nn.LSTM(
            input_size=embed_dim,
            hidden_size=enc_hidden,
            batch_first=True,
            bidirectional=True
        )

        # scalar attention over BiLSTM outputs
        self.att_linear = nn.Linear(enc_hidden * 2, 1)

        self.dropout = nn.Dropout(dropout)

        # decoder LSTM (takes attended encoder representation)
        self.decoder = nn.LSTM(
            input_size=enc_hidden * 2,
            hidden_size=dec_hidden,
            batch_first=True,
            bidirectional=False
        )

        self.classifier = nn.Linear(dec_hidden, num_labels)

        # loss ignores -100 labels (your subword masking style)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, attention_mask, labels=None):
        """
        input_ids: (B, T)
        attention_mask: (B, T) 1 for real tokens, 0 for pad
        labels: (B, T) with -100 to ignore
        """
        x = self.embedding(input_ids)  # (B, T, E)

        enc_out, _ = self.encoder(x)   # (B, T, 2H)

        # attention scores
        att_scores = self.att_linear(enc_out).squeeze(-1)               # (B, T)
        att_scores = att_scores.masked_fill(attention_mask == 0, -1e9)  # ignore pad
        att_w = torch.softmax(att_scores, dim=-1).unsqueeze(-1)         # (B, T, 1)

        # apply attention (token-wise weighting)
        att_out = enc_out * att_w                                       # (B, T, 2H)
        att_out = self.dropout(att_out)

        dec_out, _ = self.decoder(att_out)                              # (B, T, D)
        logits = self.classifier(self.dropout(dec_out))                 # (B, T, num_labels)

        if labels is not None:
            # CrossEntropy expects (B*T, C) and (B*T,)
            loss = self.loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            return loss
        else:
            preds = torch.argmax(logits, dim=-1)                        # (B, T)
            return preds


In [35]:
MODEL_TYPE = "BiLSTM-att-LSTM"

def build_model():
    vocab_size = tokenizer.vocab_size
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    model = BiLstmAttLstm(
        vocab_size=vocab_size,
        num_labels=num_labels,
        embed_dim=300,
        enc_hidden=300,
        dec_hidden=300,
        dropout=0.1,
        pad_token_id=pad_id,
    ).to(device)

    return model


In [55]:
import torch

def train_model(model, train_loader, device, lr=1e-3, epochs=40, grad_clip=1.0):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0.0

        for batch in train_loader:
            optimizer.zero_grad()

            loss = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )

            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss / len(train_loader):.4f}", flush=True)


In [56]:
model = build_model()

train_model(
    model=model,
    train_loader=train_loader,
    device=device
)


Epoch 1/40 - Loss: 1.4657
Epoch 2/40 - Loss: 1.1105
Epoch 3/40 - Loss: 0.9486
Epoch 4/40 - Loss: 0.8272
Epoch 5/40 - Loss: 0.7078
Epoch 6/40 - Loss: 0.6089
Epoch 7/40 - Loss: 0.5218
Epoch 8/40 - Loss: 0.4492
Epoch 9/40 - Loss: 0.3870
Epoch 10/40 - Loss: 0.3341
Epoch 11/40 - Loss: 0.2930
Epoch 12/40 - Loss: 0.2570
Epoch 13/40 - Loss: 0.2318
Epoch 14/40 - Loss: 0.2008
Epoch 15/40 - Loss: 0.1803
Epoch 16/40 - Loss: 0.1631
Epoch 17/40 - Loss: 0.1415
Epoch 18/40 - Loss: 0.1315
Epoch 19/40 - Loss: 0.1173
Epoch 20/40 - Loss: 0.1075
Epoch 21/40 - Loss: 0.0957
Epoch 22/40 - Loss: 0.0922
Epoch 23/40 - Loss: 0.0931
Epoch 24/40 - Loss: 0.0843
Epoch 25/40 - Loss: 0.0752
Epoch 26/40 - Loss: 0.0657
Epoch 27/40 - Loss: 0.0639
Epoch 28/40 - Loss: 0.0561
Epoch 29/40 - Loss: 0.0516
Epoch 30/40 - Loss: 0.0507
Epoch 31/40 - Loss: 0.0484
Epoch 32/40 - Loss: 0.0472
Epoch 33/40 - Loss: 0.0425
Epoch 34/40 - Loss: 0.0428
Epoch 35/40 - Loss: 0.0467
Epoch 36/40 - Loss: 0.0509
Epoch 37/40 - Loss: 0.0508
Epoch 38/4

In [59]:
from seqeval.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report
)

def evaluate(model, dataloader, device, id2label):
    model.eval()
    all_true = []
    all_pred = []

    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"]
            labels = batch["labels"]

            preds = model(input_ids=input_ids, attention_mask=attention_mask)

            for i in range(input_ids.size(0)):
                true_seq = labels[i].cpu().tolist()
                pred_seq = preds[i].cpu().tolist()
                mask_seq = attention_mask[i].cpu().tolist()

                true_tags = []
                pred_tags = []

                for t, p, m in zip(true_seq, pred_seq, mask_seq):
                    if m == 0:
                        break
                    if t == -100:
                        continue

                    true_label = id2label[t]
                    pred_label = id2label[p]

                    true_tags.append(true_label)
                    pred_tags.append(pred_label)

                    total += 1
                    if true_label == pred_label:
                        correct += 1

                all_true.append(true_tags)
                all_pred.append(pred_tags)

    accuracy = correct / total

    overall_precision = precision_score(all_true, all_pred)
    overall_recall = recall_score(all_true, all_pred)
    overall_f1 = f1_score(all_true, all_pred)

    print("=== OVERALL METRICS ===")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {overall_precision:.4f}")
    print(f"Recall:    {overall_recall:.4f}")
    print(f"F1 Score:  {overall_f1:.4f}")

    print("\n=== DETAILED CLASSIFICATION REPORT ===")
    print(classification_report(all_true, all_pred))

    return accuracy, overall_precision, overall_recall, overall_f1


In [60]:
accuracy, precision, recall, f1 = evaluate(
    model, test_loader, device, id2label
)


=== OVERALL METRICS ===
Accuracy:  0.7815
Precision: 0.4708
Recall:    0.4751
F1 Score:  0.4730

=== DETAILED CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

          AP       0.63      0.57      0.60       415
          AV       0.28      0.65      0.39        34
         CoA       0.00      0.00      0.00        11
         Com       0.57      0.55      0.56      1012
         Con       0.40      0.48      0.44       402
         Ide       0.11      0.16      0.13        55
         Loc       0.14      0.26      0.18        50
        Tool       0.41      0.20      0.27       171
         Veh       0.69      0.32      0.44       149
         Vul       0.19      0.30      0.23       130

   micro avg       0.47      0.48      0.47      2429
   macro avg       0.34      0.35      0.32      2429
weighted avg       0.50      0.48      0.48      2429



In [61]:
import pandas as pd
import random
import numpy as np
import torch

SEEDS = list(range(10))
results = []

for seed in SEEDS:
    print(f"\nRunning seed {seed}")

    # set seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # build + train model
    model = build_model()

    train_model(
        model=model,
        train_loader=train_loader,
        device=device
    )

    # evaluate
    accuracy, precision, recall, f1 = evaluate(
        model, test_loader, device, id2label
    )

    results.append({
        "Seed": seed,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

# results table
df = pd.DataFrame(results)
print("\n=== SEED-WISE RESULTS ===")
print(df)

print("\n=== AVERAGE RESULTS (10 seeds) ===")
print(df.mean(numeric_only=True))



Running seed 0
Epoch 1/40 - Loss: 1.4649
Epoch 2/40 - Loss: 1.1034
Epoch 3/40 - Loss: 0.9405
Epoch 4/40 - Loss: 0.8333
Epoch 5/40 - Loss: 0.7320
Epoch 6/40 - Loss: 0.6350
Epoch 7/40 - Loss: 0.5601
Epoch 8/40 - Loss: 0.4914
Epoch 9/40 - Loss: 0.4261
Epoch 10/40 - Loss: 0.3763
Epoch 11/40 - Loss: 0.3372
Epoch 12/40 - Loss: 0.2952
Epoch 13/40 - Loss: 0.2617
Epoch 14/40 - Loss: 0.2426
Epoch 15/40 - Loss: 0.2205
Epoch 16/40 - Loss: 0.1986
Epoch 17/40 - Loss: 0.1774
Epoch 18/40 - Loss: 0.1661
Epoch 19/40 - Loss: 0.1399
Epoch 20/40 - Loss: 0.1338
Epoch 21/40 - Loss: 0.1201
Epoch 22/40 - Loss: 0.1154
Epoch 23/40 - Loss: 0.1052
Epoch 24/40 - Loss: 0.0927
Epoch 25/40 - Loss: 0.0838
Epoch 26/40 - Loss: 0.0846
Epoch 27/40 - Loss: 0.0826
Epoch 28/40 - Loss: 0.0741
Epoch 29/40 - Loss: 0.0704
Epoch 30/40 - Loss: 0.0693
Epoch 31/40 - Loss: 0.0640
Epoch 32/40 - Loss: 0.0619
Epoch 33/40 - Loss: 0.0608
Epoch 34/40 - Loss: 0.0586
Epoch 35/40 - Loss: 0.0574
Epoch 36/40 - Loss: 0.0544
Epoch 37/40 - Loss: 0